### 🖼️ Side-by-Side Comparison: Noisy LR vs Predicted HR

This cell decodes predictions from `submission.csv` and loads the corresponding original noisy images from `test/NoisyLR/` to display them side-by-side using Matplotlib.

In [5]:
import pandas as pd
import numpy as np
import base64
import io
import os
import matplotlib.pyplot as plt

def decode_npy_base64(b64_str):
    """Decode base64 string back to NumPy array."""
    buffer = io.BytesIO(base64.b64decode(b64_str))
    return np.load(buffer)

def plot_comparison(submission_path='submission.csv', noisy_dir='test/NoisyLR', num_images=5, start_index=0):
    # Load submission
    df = pd.read_csv(submission_path)
    
    fig, axes = plt.subplots(num_images, 2, figsize=(10, 4 * num_images))
    if num_images == 1:
        axes = np.expand_dims(axes, axis=0)
        
    for i in range(num_images):
        idx = start_index + i
        if idx >= len(df):
            break
            
        row = df.iloc[idx]
        img_id = int(row['id'])
        b64_str = row['npy_base64']
        
        # Decode prediction
        try:
            pred_img = decode_npy_base64(b64_str)
        except Exception as e:
            print(f"Error decoding id {img_id}: {e}")
            continue
            
        # Map 1-based submission ID to 0-based filename filename
        # (Match: ID 1 -> 000000.npy)
        noisy_filename = os.path.join(noisy_dir, f"{img_id-1:06d}.npy")
        
        # Load noisy image
        if os.path.exists(noisy_filename):
            noisy_img = np.load(noisy_filename)
        else:
            print(f"File not found: {noisy_filename}")
            noisy_img = np.zeros_like(pred_img) # placeholder
            
        # Plotting
        axes[i, 0].imshow(noisy_img, cmap='gray')
        axes[i, 0].set_title(f"Noisy LR (File: {img_id-1:06d}.npy)")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(pred_img, cmap='gray')
        axes[i, 1].set_title(f"Submission Prediction (ID: {img_id})")
        axes[i, 1].axis('off')
        
    plt.tight_layout()
    plt.show()

# Usage: Plot first 5 comparisons
plot_comparison(num_images=3, start_index=0)

ModuleNotFoundError: No module named 'pandas'